In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
from utils.metric import metric

pd.set_option("display.max_columns", 100)

In [2]:
train = pd.read_csv('../../data/train.csv')

In [3]:
HORIZONS = [12, 24, 48, 72]

In [4]:
def make_strata(df):
    conds = [
        (df.event == 0),
        (df.event == 1) & (df.time_to_hit_hours <= 12),
        (df.event == 1) & (df.time_to_hit_hours <= 24),
        (df.event == 1) & (df.time_to_hit_hours <= 48),
        (df.event == 1)
    ]
    return np.select(conds, [0,1,2,3,4])

In [5]:
train["strata"] = make_strata(train)

In [6]:
train.head()

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,log_area_ratio_0_5h,relative_growth_0_5h,radial_growth_m,radial_growth_rate_m_per_h,centroid_displacement_m,centroid_speed_m_per_h,spread_bearing_deg,spread_bearing_sin,spread_bearing_cos,dist_min_ci_0_5h,dist_std_ci_0_5h,dist_change_ci_0_5h,dist_slope_ci_0_5h,closing_speed_m_per_h,closing_speed_abs_m_per_h,projected_advance_m,dist_accel_m_per_h2,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event,strata
0,20307683,1,0.000000,1,37.726426,0.000000,0.000000,0.000000,3.656522,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,756438.281342,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,21,1,3,57.643807,0,0
1,60132804,13,4.819051,0,1630.389625,2508.041442,1.538308,520.443033,7.397187,7.827656,0.931498,1.538308,1351.378212,280.424145,2045.333109,424.426546,93.897804,0.997687,-0.067977,2798.867829,850.735044,-1706.526611,-566.710918,354.120897,354.120897,1706.526611,-116.927386,0.502981,0.902628,0.902628,-182.682530,383.099186,20,2,7,1.262348,1,1
2,90380661,1,0.000000,1,68.796478,0.000000,0.000000,0.000000,4.245584,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,158184.073387,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,15,2,8,46.530571,0,0
3,49907151,12,4.829815,0,200.386653,314.052044,1.567230,65.023617,5.305227,5.752738,0.942828,1.567230,480.996547,99.589019,654.749371,135.564066,59.998434,0.866012,0.500024,2430.597799,325.137571,-650.275319,-202.202867,134.637726,134.637726,650.275319,-40.216195,0.638522,0.994594,0.994594,14.077093,134.831196,20,3,8,0.508562,1,1
4,18104294,1,0.000000,1,147.357925,0.000000,0.000000,0.000000,4.999628,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1742.764246,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14,1,9,23.843843,1,2


In [7]:
def expand_discrete(df, features):
    rows = []

    for idx, r in df.iterrows():
        for h in HORIZONS:

            if r.event == 0 and r.time_to_hit_hours < h:
                break

            if r.event == 1 and r.time_to_hit_hours <= h:
                y = 1
                row = r[features].to_dict()
                row["horizon"] = h
                row["y"] = y
                row["id"] = idx
                rows.append(row)
                break

            y = 0
            row = r[features].to_dict()
            row["horizon"] = h
            row["y"] = y
            row["id"] = idx
            rows.append(row)

    return pd.DataFrame(rows)

In [10]:
features = [c for c in train.columns if c not in 
            ["event","time_to_hit_hours","strata"]]

skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

oof = pd.DataFrame(
    0.0,
    index=train.index,
    columns=[f"p{h}" for h in HORIZONS]
)

In [16]:
for fold, (tr_idx, val_idx) in enumerate(skf.split(train, train["strata"])):

    tr = train.iloc[tr_idx]
    val = train.iloc[val_idx]

    X_tr = tr[features]
    X_val = val[features]

    for H in HORIZONS:
        y_tr = ((tr.event == 1) & (tr.time_to_hit_hours <= H)).astype(int)
        y_val = ((val.event == 1) & (val.time_to_hit_hours <= H)).astype(int)

        clf = lgb.LGBMClassifier(
            n_estimators=600,
            learning_rate=0.03,
            num_leaves=31,
            min_child_samples=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42 + fold,
            verbosity=-1
        )

        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_val)[:, 1]

        iso = IsotonicRegression(out_of_bounds="clip")
        p_cal = iso.fit_transform(p, y_val)

        oof.loc[val.index, f"p{H}"] = p_cal

oof["p24"] = np.maximum(oof["p24"], oof["p12"])
oof["p48"] = np.maximum(oof["p48"], oof["p24"])
oof["p72"] = np.maximum(oof["p72"], oof["p48"])

probs = oof[[f"p{h}" for h in HORIZONS]].values
time_to_hit = train["time_to_hit_hours"].values
event = train["event"].values

c_index, brier, final_score = metric(probs, time_to_hit, event)

print("C-index:", c_index)
print("Brier:", brier)
print("Final:", final_score)

C-index: 0.9366916639495707
Brier: 0.019440158170053085
Final: 0.9673993884658341


In [18]:
val = pd.read_csv("../../data/val.csv")

final_preds = pd.DataFrame(
    0.0,
    index=val.index,
    columns=[f"p{h}" for h in HORIZONS]
)

for H in HORIZONS:

    y_full = ((train.event == 1) &
              (train.time_to_hit_hours <= H)).astype(int)

    clf = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=15,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        random_state=42,
        verbosity=-1
    )

    clf.fit(train[features], y_full)

    p_train = clf.predict_proba(train[features])[:,1]

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_train, y_full)

    p_val = clf.predict_proba(val[features])[:,1]
    p_val_cal = iso.transform(p_val)

    final_preds[f"p{H}"] = p_val_cal

In [19]:
final_preds["p24"] = np.maximum(final_preds["p24"], final_preds["p12"])
final_preds["p48"] = np.maximum(final_preds["p48"], final_preds["p24"])
final_preds["p72"] = np.maximum(final_preds["p72"], final_preds["p48"])

In [20]:
probs_val = final_preds[[f"p{h}" for h in HORIZONS]].values

c_index, brier, final_score = metric(
    probs_val,
    val["time_to_hit_hours"].values,
    val["event"].values
)

print("VAL C-index:", c_index)
print("VAL Brier:", brier)
print("VAL Final:", final_score)

VAL C-index: 0.9166666666666666
VAL Brier: 0.05119516074340638
VAL Final: 0.9391633874796155
